In [35]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from transformer import Transformer
from util import CharDataset, SinusoidalPositionalEncoding

# Preparing dataset

In [36]:
from torch.utils.data import DataLoader, DistributedSampler
# 1) Load the file you downloaded from the browser
with open("./input.txt", "r", encoding="utf-8") as f:
    text = f.read()

# 2) Train/val split (standard simple split)
  # for faster experimentation, we use only first 10k characters
n = int(0.9 * len(text))
train_text = text[:n]
val_text = text[n:]

# 3) Make datasets + loaders
block_size = 512   # context length
batch_size = 64

train_ds = CharDataset(train_text, block_size)
val_ds   = CharDataset(val_text, block_size)


# Building Model 

In [37]:
class Transformer_model(nn.Module):
    def __init__(self, num_heads, d_head, dk, dv):
        super().__init__()
        self.d_model = num_heads * d_head
        self.token_emb = nn.Embedding(train_ds.vocab_size, dk)
        self.pos_emb = SinusoidalPositionalEncoding(dk)
        self.transformer = Transformer(dk, dv, num_heads, d_head, num_encoder_layers=6, num_decoder_layers=6, output_dim=train_ds.vocab_size)
    def forward(self, x):
        # x: (B, T)
        x = self.token_emb(x) # (B, T, d_model)
        x = self.pos_emb(x)   # (B, T, d_model)
        # print("Shape after pos emb:", x.shape)
        out = self.transformer(x) # (B, T, vocab_size)
        return out

# Training 

In [38]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [39]:
import os
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler

def setup_ddp():
    dist.init_process_group(backend="nccl")
    local_rank = int(os.environ["LOCAL_RANK"])
    torch.cuda.set_device(local_rank)
    return local_rank

def cleanup_ddp():
    dist.destroy_process_group()

In [40]:
import time
import torch
import torch.nn.functional as F

local_rank = setup_ddp()
device = torch.device(f"cuda:{local_rank}")
model = Transformer_model(num_heads=8, d_head=64, dk=512, dv=512)

model = model.to(device)
model = DDP(model, device_ids=[local_rank], output_device=local_rank)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

num_epochs = 5


train_sampler = DistributedSampler(train_ds, shuffle=True)
val_sampler = DistributedSampler(val_ds, shuffle=False)

print(f"train has {len(train_ds)} samples")

train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=train_sampler, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=batch_size, sampler=val_sampler, drop_last=True)

print("vocab_size:", train_ds.vocab_size)
x, y = next(iter(train_loader))
print("x shape:", x.shape, "y shape:", y.shape)  # (B, T), (B, T)

def train_step(x, y):
    x = x.to(device)  # (B, T)
    y = y.to(device)  # (B, T)

    logits = model(x)  # (B, T, vocab_size)

    loss = F.cross_entropy(
        logits.view(-1, train_ds.vocab_size),
        y.view(-1)
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()


@torch.no_grad()
def evaluate():
    model.eval()
    total_loss = 0.0

    for x, y in val_loader:
        x = x.to(device)
        y = y.to(device)

        logits = model(x)
        loss = F.cross_entropy(
            logits.view(-1, train_ds.vocab_size),
            y.view(-1)
        )
        total_loss += loss.item()

    return total_loss / len(val_loader)


for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0
    t0 = time.perf_counter()
    for i, (x, y) in enumerate(train_loader):
       
        
        loss = train_step(x, y)                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             
        total_loss += loss
       
        if i % 100 == 0:
            dt = time.perf_counter() - t0
            print(f"Epoch {epoch + 1} | Step {i + 1} | Loss {loss:.4f} | Time per steps: {(dt*1000/(i+1)):.2f} ms")

    avg_train_loss = total_loss / len(train_loader)                                                        
    

    print(f"Epoch {epoch+1} | Train loss: {avg_train_loss:.4f}")
    val_loss = evaluate()
    print(f"Epoch {epoch+1} | Val loss: {val_loss:.4f}")



ValueError: Error initializing torch.distributed using env:// rendezvous: environment variable RANK expected, but not set